In [ ]:
import torch

data = torch.load("../activations/bbq.pt", weights_only=False)

# SAE activations are stored as sparse tensors — convert to dense
sae_activations = [act.to_dense() for act in data["sae_activations"]]
generations     = data["generations"]
categories      = data["categories"]
model_config    = data["model_config"]
sae_config      = data["sae_config"]

print(f"Loaded {len(sae_activations)} samples")
print(f"Model: {model_config['model_name']}")
print(f"SAE:   layer {sae_config['layer']}, width {sae_config['width']}, L0 {sae_config['l0']}")

In [ ]:
from src.aggregator import Aggregator

aggregator = Aggregator()

# List of (d_sae,) tensors, one per prompt
aggregated = [aggregator.max(act) for act in sae_activations]

# Stack into (n_samples, d_sae)
aggregated_matrix = torch.stack(aggregated)
print(f"Aggregated matrix shape: {aggregated_matrix.shape}")

In [ ]:
from src.denoiser import Denoiser

denoiser = Denoiser()

# (n_samples, d_sae) → (n_samples, d_sae), z-scored per feature across prompts
normalised_matrix = denoiser.global_idf(aggregated_matrix)
print(f"Normalised matrix shape: {normalised_matrix.shape}")

In [ ]:
from src.neuronpedia_client import NeuronpediaClient, build_sae_id
from src.configs import SAEConfig
from src.feature import Feature

PROMPT_IDX = 827  # change to inspect a different prompt
TOP_K = 20

prompt_vec = normalised_matrix[PROMPT_IDX]          # (d_sae,)
top_strengths, top_indices = prompt_vec.topk(TOP_K)

# Build Neuronpedia client
model_id = model_config["model_name"].split("/")[-1]
sae_cfg = SAEConfig(
    repo_id=sae_config["repo_id"],
    sae_type=sae_config["sae_type"],
    layer=sae_config["layer"],
    width=sae_config["width"],
    l0=sae_config["l0"],
)
client = NeuronpediaClient(model_id=model_id, sae_id=build_sae_id(sae_cfg))

features = Feature.from_activations(top_indices, top_strengths, client)

print(f"Prompt #{PROMPT_IDX}  |  category: {categories[PROMPT_IDX]}")
print(f"\nGeneration:\n{generations[PROMPT_IDX]}\n")
print(f"Top {TOP_K} SAE features (z-scored strength):")
for f in features:
    desc = f.description or "(no description)"
    print(f"  Feature {f.feature_idx:>6d}  z={f.strength:+.3f}  ->  {desc}")